In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [23]:
import os
import cv2
import time
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import tensorflow as tf

In [3]:
SOURCE_DIR = "/content/drive/MyDrive/Colab Notebooks/Archive/test"
TARGET_DIR = "/content/drive/MyDrive/Colab Notebooks/Archive/test_dark"

os.makedirs(TARGET_DIR, exist_ok=True)

for cls in os.listdir(SOURCE_DIR):

    src_cls = os.path.join(SOURCE_DIR, cls)

    if not os.path.isdir(src_cls):
        continue

    dst_cls = os.path.join(TARGET_DIR, cls)

    os.makedirs(dst_cls, exist_ok=True)

    for img_name in os.listdir(src_cls):

        img_path = os.path.join(src_cls, img_name)

        img = cv2.imread(img_path)

        dark = cv2.convertScaleAbs(
            img,
            alpha=0.7,
            beta=-40
        )

        cv2.imwrite(
            os.path.join(dst_cls, img_name),
            dark
        )

print("Dark dataset created.")

Dark dataset created.


In [4]:
SOURCE_DIR = "/content/drive/MyDrive/Colab Notebooks/Archive/test"
TARGET_DIR = "/content/drive/MyDrive/Colab Notebooks/Archive/test_rotate"

os.makedirs(TARGET_DIR, exist_ok=True)

for cls in os.listdir(SOURCE_DIR):

    src_cls = os.path.join(SOURCE_DIR, cls)
    dst_cls = os.path.join(TARGET_DIR, cls)

    if not os.path.isdir(src_cls):
        continue

    os.makedirs(dst_cls, exist_ok=True)

    for img_name in os.listdir(src_cls):

        img = cv2.imread(
            os.path.join(src_cls, img_name)
        )

        h, w = img.shape[:2]

        M = cv2.getRotationMatrix2D(
            (w/2, h/2),
            30,
            1.0
        )

        rotated = cv2.warpAffine(
            img,
            M,
            (w, h)
        )

        cv2.imwrite(
            os.path.join(dst_cls, img_name),
            rotated
        )

print("Rotate dataset created.")

Rotate dataset created.


In [5]:
SOURCE_DIR = "/content/drive/MyDrive/Colab Notebooks/Archive/test"
TARGET_DIR = "/content/drive/MyDrive/Colab Notebooks/Archive/test_blur"

os.makedirs(TARGET_DIR, exist_ok=True)

for cls in os.listdir(SOURCE_DIR):

    src_cls = os.path.join(SOURCE_DIR, cls)
    dst_cls = os.path.join(TARGET_DIR, cls)

    if not os.path.isdir(src_cls):
        continue

    os.makedirs(dst_cls, exist_ok=True)

    for img_name in os.listdir(src_cls):

        img = cv2.imread(
            os.path.join(src_cls, img_name)
        )

        blur = cv2.GaussianBlur(
            img,
            (7,7),
            0
        )

        cv2.imwrite(
            os.path.join(dst_cls, img_name),
            blur
        )

print("Blur dataset created.")

Blur dataset created.


## Load Models

In [36]:
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/Archive"

cnn_model = tf.keras.models.load_model(
    f"{BASE_DIR}/CNN_Baseline.keras"
)

cnn_aug_model = tf.keras.models.load_model(
    f"{BASE_DIR}/CNN_Augmentation.keras"
)

resnet_model = tf.keras.models.load_model(
    f"{BASE_DIR}/ResNet50.keras"
)

efficientnet_model = tf.keras.models.load_model(
    f"{BASE_DIR}/EfficientNetB0.keras"
)

vit_model = tf.keras.models.load_model(
    f"{BASE_DIR}/ViT.keras"
)

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 402 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [7]:
models = {
    "CNN": cnn_model,
    "CNN+Aug": cnn_aug_model,
    "ResNet50": resnet_model,
    "EfficientNetB0": efficientnet_model,
    "ViT": vit_model
}

## Create Datasets

In [8]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

In [9]:
test_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/drive/MyDrive/Colab Notebooks/Archive/test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

Found 100 files belonging to 20 classes.


In [10]:
test_dark_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/drive/MyDrive/Colab Notebooks/Archive/test_dark",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

Found 100 files belonging to 20 classes.


In [11]:
test_blur_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/drive/MyDrive/Colab Notebooks/Archive/test_blur",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

Found 100 files belonging to 20 classes.


In [12]:
test_rotate_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/drive/MyDrive/Colab Notebooks/Archive/test_rotate",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

Found 100 files belonging to 20 classes.


In [13]:
datasets = {
    "Original": test_ds,
    "Dark": test_dark_ds,
    "Blur": test_blur_ds,
    "Rotate": test_rotate_ds
}

## Evaluate Models

In [14]:
results = []

for model_name, model in models.items():

    print(f"\n===== {model_name} =====")

    for dataset_name, ds in datasets.items():

        print(f"Evaluating {dataset_name}")

        loss, acc = model.evaluate(
            ds,
            verbose=0
        )

        print(f"Accuracy = {acc:.4f}")

        results.append(
            [model_name, dataset_name, acc]
        )


===== CNN =====
Evaluating Original
Accuracy = 0.7700
Evaluating Dark
Accuracy = 0.6600
Evaluating Blur
Accuracy = 0.5300
Evaluating Rotate
Accuracy = 0.4200

===== CNN+Aug =====
Evaluating Original
Accuracy = 0.8800
Evaluating Dark
Accuracy = 0.7000
Evaluating Blur
Accuracy = 0.7000
Evaluating Rotate
Accuracy = 0.8200

===== ResNet50 =====
Evaluating Original
Accuracy = 0.9600
Evaluating Dark
Accuracy = 0.9300
Evaluating Blur
Accuracy = 0.9200
Evaluating Rotate
Accuracy = 0.9600

===== EfficientNetB0 =====
Evaluating Original
Accuracy = 0.9700
Evaluating Dark
Accuracy = 0.9800
Evaluating Blur
Accuracy = 0.9200
Evaluating Rotate
Accuracy = 0.9700

===== ViT =====
Evaluating Original
Accuracy = 0.9400
Evaluating Dark
Accuracy = 0.9200
Evaluating Blur
Accuracy = 0.9700
Evaluating Rotate
Accuracy = 0.9500


In [15]:
df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Dataset",
        "Accuracy"
    ]
)

print(df)

             Model   Dataset  Accuracy
0              CNN  Original      0.77
1              CNN      Dark      0.66
2              CNN      Blur      0.53
3              CNN    Rotate      0.42
4          CNN+Aug  Original      0.88
5          CNN+Aug      Dark      0.70
6          CNN+Aug      Blur      0.70
7          CNN+Aug    Rotate      0.82
8         ResNet50  Original      0.96
9         ResNet50      Dark      0.93
10        ResNet50      Blur      0.92
11        ResNet50    Rotate      0.96
12  EfficientNetB0  Original      0.97
13  EfficientNetB0      Dark      0.98
14  EfficientNetB0      Blur      0.92
15  EfficientNetB0    Rotate      0.97
16             ViT  Original      0.94
17             ViT      Dark      0.92
18             ViT      Blur      0.97
19             ViT    Rotate      0.95


In [16]:
pivot_df = df.pivot(
    index="Model",
    columns="Dataset",
    values="Accuracy"
)

pivot_df = pivot_df[
    ["Original", "Dark", "Blur", "Rotate"]
]

print(pivot_df)

Dataset         Original  Dark  Blur  Rotate
Model                                       
CNN                 0.77  0.66  0.53    0.42
CNN+Aug             0.88  0.70  0.70    0.82
EfficientNetB0      0.97  0.98  0.92    0.97
ResNet50            0.96  0.93  0.92    0.96
ViT                 0.94  0.92  0.97    0.95


In [17]:
drop_df = pd.DataFrame(index=pivot_df.index)

drop_df["Dark Drop (%)"] = (
    (pivot_df["Original"] - pivot_df["Dark"])
    / pivot_df["Original"]
    * 100
)

drop_df["Blur Drop (%)"] = (
    (pivot_df["Original"] - pivot_df["Blur"])
    / pivot_df["Original"]
    * 100
)

drop_df["Rotate Drop (%)"] = (
    (pivot_df["Original"] - pivot_df["Rotate"])
    / pivot_df["Original"]
    * 100
)

drop_df = drop_df.round(2)

print(drop_df)

                Dark Drop (%)  Blur Drop (%)  Rotate Drop (%)
Model                                                        
CNN                     14.29          31.17            45.45
CNN+Aug                 20.45          20.45             6.82
EfficientNetB0          -1.03           5.15             0.00
ResNet50                 3.12           4.17             0.00
ViT                      2.13          -3.19            -1.06


In [41]:
# ==========================================================
# Model Efficiency Evaluation
# ==========================================================

import os
import time
import numpy as np
import pandas as pd

# ----------------------------------------------------------
# Model file mapping
# ----------------------------------------------------------
MODEL_FILES = {
    "CNN": "CNN_Baseline.keras",
    "CNN+Aug": "CNN_Augmentation.keras",
    "ResNet50": "ResNet50.keras",
    "EfficientNetB0": "EfficientNetB0.keras",
    "ViT": "ViT.keras"
}

# ----------------------------------------------------------
# Use ONE image for latency benchmark
# ----------------------------------------------------------
images, _ = next(iter(test_ds))
images = images[:1]        # Batch size = 1

N_WARMUP = 10
N_REPEAT = 100

results = []

print("=" * 70)
print("Running Model Efficiency Evaluation...")
print("=" * 70)

for name, model in models.items():

    print(f"Testing {name}...")

    # ------------------------------------------------------
    # Number of Parameters
    # ------------------------------------------------------
    params = model.count_params() / 1e6

    # ------------------------------------------------------
    # Model Size
    # ------------------------------------------------------
    model_size = os.path.getsize(
        os.path.join(BASE_DIR, MODEL_FILES[name])
    ) / (1024 * 1024)

    # ------------------------------------------------------
    # Warm-up
    # ------------------------------------------------------
    for _ in range(N_WARMUP):
        _ = model(images, training=False)

    # ------------------------------------------------------
    # Measure inference time
    # ------------------------------------------------------
    inference_times = []

    for _ in range(N_REPEAT):

        start = time.perf_counter()

        _ = model(images, training=False)

        end = time.perf_counter()

        inference_times.append(end - start)

    avg_time = np.mean(inference_times) * 1000      # ms
    std_time = np.std(inference_times) * 1000       # ms

    results.append({
        "Model": name,
        "Parameters (M)": round(params, 2),
        "Model Size (MB)": round(model_size, 2),
        "Average Inference Time (ms/image)": round(avg_time, 2)
    })

# ----------------------------------------------------------
# Create dataframe
# ----------------------------------------------------------
efficiency_df = pd.DataFrame(results)

# Rename model names for the dissertation
efficiency_df["Model"] = efficiency_df["Model"].replace({
    "CNN": "Baseline CNN",
    "CNN+Aug": "CNN (Data Augmentation)",
    "EfficientNetB0": "EfficientNet-B0"
})

display(efficiency_df)

# ----------------------------------------------------------
# Save CSV
# ----------------------------------------------------------
efficiency_df.to_csv(
    "ModelEfficiencyComparison.csv",
    index=False
)

print("CSV saved as ModelEfficiencyComparison.csv")

Running Model Efficiency Evaluation...
Testing CNN...
Testing CNN+Aug...
Testing ResNet50...
Testing EfficientNetB0...
Testing ViT...


,Model,Parameters (M),Model Size (MB),Average Inference Time (ms/image)
0,Baseline CNN,11.17,127.90,9.46
1,CNN (Data Augmentation),11.17,127.91,10.80
2,ResNet50,23.63,91.09,235.63
3,EfficientNet-B0,4.08,16.56,293.56
4,ViT,85.81,327.93,204.75


CSV saved as ModelEfficiencyComparison.csv
